# Aprendizado de Máquina — Aula prática 07

## Classificação e Classificadores Gaussianos

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Até aqui $Y$ era um número e o erro era quadrático. A partir de agora $Y$ é uma
**classe**, e quase tudo precisa ser reescrito — o risco, o estimador ótimo, as
métricas. A boa notícia é que a estrutura é a mesma, e a peça central continua
sendo uma esperança condicional:

> **classificar bem é estimar bem $P(Y=1 \mid X=x)$, e depois decidir.**

Repare que são **duas** coisas, e a Aula 08 inteira vive dessa separação. Neste
notebook cuidamos da primeira: três maneiras de estimar essa probabilidade — as três
pelo Teorema de Bayes —, o que cada uma supõe, e o que cada suposição custa. E, como sempre, vamos trabalhar numa
população inventada, onde a probabilidade verdadeira é conhecida — o que nos deixa
medir o **piso** que nenhum classificador pode furar.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- calcular o classificador de Bayes e o **erro de Bayes** numa população conhecida;
- ajustar Bayes ingênuo, LDA e QDA, e explicar a forma da fronteira de cada um a
  partir do que ele supõe;
- ler `predict_proba` e reconhecer que o corte em $0{,}5$ é uma escolha, não uma
  consequência;
- medir a troca viés–variância entre LDA e QDA em função de $n$ **e de $p$**;
- aplicar os três a um banco real e ler o resultado sem confundir ruído com sinal.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são os três classificadores da aula e a `multivariate_normal`
do `scipy`, que dá a densidade exata das gaussianas que vamos inventar — é com ela
que calculamos o classificador de Bayes.

In [ ]:
import sklearn.model_selection as skm
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis as LDA,
                                           QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB

from scipy.stats import multivariate_normal

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A população, e o classificador de Bayes

Duas classes gaussianas em $\mathbb{R}^2$, com médias e **covariâncias diferentes**. Como
somos nós que inventamos os parâmetros, podemos escrever a regra ótima.

O classificador de Bayes prevê a classe mais provável dado $x$:

$$g^*(x) = \arg\max_c \; P(Y=c \mid X=x)
        = \arg\max_c \; \pi_c \, f_c(x),$$

onde $\pi_c$ é a proporção da classe e $f_c$ a densidade dela. Nenhum
classificador tem risco menor que o dele — é o análogo do $\sigma^2$ da Aula 01.

In [ ]:
S0 = np.array([[1.00, 0.75], [0.75, 1.00]])
S1 = np.array([[1.60, -0.85], [-0.85, 0.70]])
mu0, mu1 = np.array([0.0, 0.0]), np.array([2.2, 2.2 * 0.35])
pi0 = pi1 = 0.5


def amostra(n, rng):
    n0 = n // 2
    X = np.vstack([rng.multivariate_normal(mu0, S0, size=n0),
                   rng.multivariate_normal(mu1, S1, size=n - n0)])
    y = np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)
    return X, y


def prob_bayes(X):
    """P(Y=1 | X=x), exata -- so existe porque inventamos a populacao."""
    f0 = pi0 * multivariate_normal(mu0, S0).pdf(X)
    f1 = pi1 * multivariate_normal(mu1, S1).pdf(X)
    return f1 / (f0 + f1)


rng = np.random.default_rng(31)
X, y = amostra(400, rng)
X_te, y_te = amostra(40_000, np.random.default_rng(99))

erro_bayes = np.mean((prob_bayes(X_te) > 0.5).astype(int) != y_te)
print(f"erro do classificador de Bayes: {erro_bayes:.4f}")
print(f"acuracia maxima possivel      : {1 - erro_bayes:.4f}")
print(f"erro de quem chuta a classe majoritaria: {min(y_te.mean(), 1-y_te.mean()):.4f}")

Esse número é o piso. Nenhum dos classificadores que virão adiante pode ficar
abaixo dele, por mais dados que receba — as duas classes se sobrepõem, e a
sobreposição é irredutível. É exatamente o papel que $\sigma^2$ fazia em regressão.

---
## 3. Três plug-ins, três fronteiras

Como $P(Y=1\mid x)$ é desconhecida na vida real, estimamos e substituímos. É o
**princípio plug-in**. Os três métodos desta aula estimam essa probabilidade pelo
Teorema de Bayes — modelam a densidade $f_c(x)$ de cada classe e invertem — e
diferem só no modelo que usam para $f_c$:

- **Bayes ingênuo** (`GaussianNB`) supõe que as covariáveis são **independentes
  dentro de cada classe**, cada uma gaussiana;
- **LDA** modela cada classe como gaussiana, com **uma covariância comum**;
- **QDA** modela cada classe como gaussiana, com **covariância própria**.

In [ ]:
modelos = [("Bayes ingenuo", GaussianNB()),
           ("LDA (cov. comum)", LDA()),
           ("QDA (cov. por classe)", QDA())]

xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .8, X[:, 0].max() + .8, 320),
                     np.linspace(X[:, 1].min() - .8, X[:, 1].max() + .8, 320))
grade = np.c_[xx.ravel(), yy.ravel()]
Z_bayes = (prob_bayes(grade) > 0.5).reshape(xx.shape)

fig, axes = subplots(1, 3, figsize=(8.4, 3.0), sharex=True, sharey=True)
for ax, (nome, m) in zip(axes, modelos):
    m.fit(X, y)
    Z = m.predict_proba(grade)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[0, 0.5, 1], colors=["steelblue", "crimson"],
                alpha=0.10)
    ax.contour(xx, yy, Z_bayes, levels=[0.5], colors="green", linewidths=1.6,
               linestyles="--")
    ax.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=1.3)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=6, color="steelblue", alpha=0.6)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=6, color="crimson", alpha=0.6)
    ax.set_title(nome, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
axes[0].set_ylabel("tracejada verde = fronteira de Bayes", fontsize=8);

In [ ]:
linhas = []
for nome, m in modelos:
    erro = 1 - m.fit(X, y).score(X_te, y_te)
    linhas.append({"classificador": nome, "erro no teste": erro,
                   "excesso sobre Bayes": erro - erro_bayes})
linhas.append({"classificador": "Bayes (inatingivel)", "erro no teste": erro_bayes,
               "excesso sobre Bayes": 0.0})
pd.DataFrame(linhas).set_index("classificador").round(4)

O QDA chega praticamente ao piso, e não é sorte: **ele é o modelo correto**. Nós
geramos os dados de duas gaussianas com covariâncias diferentes, que é exatamente o
que o QDA supõe. Quando a suposição do método é a verdade, o plug-in é ótimo.

Os outros dois pagam pelo que supõem errado, e cada erro tem uma assinatura
geométrica visível na figura: o LDA traça uma **reta**, porque a covariância comum
não o deixa curvar a fronteira; o Bayes ingênuo curva, mas de um jeito próprio,
porque acredita que as duas coordenadas são independentes.

---
## 4. O que cada suposição faz com as covariâncias

Dá para olhar por dentro. O LDA e o QDA estimam covariâncias explicitamente; o
Bayes ingênuo estima uma covariância **diagonal** (por supor independência). Vamos
comparar com a verdade.

In [ ]:
qda_aj = QDA(store_covariance=True).fit(X, y)
lda_aj = LDA(store_covariance=True).fit(X, y)
nb_aj = GaussianNB().fit(X, y)

for k, S in [(0, S0), (1, S1)]:
    print(f"classe {k}, covariancia VERDADEIRA:\n", S)
    print(f"classe {k}, estimada pelo QDA:\n", qda_aj.covariance_[k].round(3), "\n")
print("estimada pelo LDA (a MESMA para as duas classes):\n",
      lda_aj.covariance_.round(3))
print("\nimplicita no Bayes ingenuo (classe 0), so a diagonal:\n",
      np.diag(nb_aj.var_[0]).round(3))
print(f"\ncorrelacao verdadeira na classe 0: {S0[0,1]/np.sqrt(S0[0,0]*S0[1,1]):+.3f}")
print(f"correlacao verdadeira na classe 1: {S1[0,1]/np.sqrt(S1[0,0]*S1[1,1]):+.3f}")
print("correlacao que o Bayes ingenuo assume: 0 (por construcao)")

Tudo o que a figura mostrou está aqui em números.

O **QDA** recupera as duas covariâncias, cada uma com o seu sinal de correlação
($+0{,}75$ numa classe, $-0{,}80$ na outra). Por isso a fronteira dele acompanha a
de Bayes.

O **LDA** é obrigado a produzir **uma** matriz para as duas classes, e o que ele
devolve é uma média ponderada das duas — que não descreve nenhuma delas. É essa
imposição que torna a fronteira linear: com covariância comum, os termos
quadráticos se cancelam na diferença dos logaritmos das densidades.

O **Bayes ingênuo** joga fora os termos fora da diagonal, isto é, afirma que as
duas correlações são zero quando na verdade são $+0{,}75$ e $-0{,}80$. É a
suposição mais falsa das três, e ele é o último da tabela da Seção 3 — mas por
pouco: fica meio ponto percentual atrás do LDA ($0{,}1320$ contra $0{,}1263$ de
erro), que também erra as covariâncias, só que de outro jeito.

Toda a Seção 3 foi montada com covariâncias **diferentes**, que é a hipótese do
QDA. Vale ver o outro extremo: as duas classes com a mesma covariância, que é a
hipótese do LDA. Refazemos a figura e a tabela de erro com $\Sigma_1 = \Sigma_0$.

In [ ]:
S1_igual = S0.copy()


def amostra_igual(n, rng_a):
    n0 = n // 2
    Xa = np.vstack([rng_a.multivariate_normal(mu0, S0, size=n0),
                    rng_a.multivariate_normal(mu1, S1_igual, size=n - n0)])
    return Xa, np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)


def prob_bayes_igual(Xa):
    f0 = pi0 * multivariate_normal(mu0, S0).pdf(Xa)
    f1 = pi1 * multivariate_normal(mu1, S1_igual).pdf(Xa)
    return f1 / (f0 + f1)


Xi, yi = amostra_igual(400, np.random.default_rng(31))
Xi_te, yi_te = amostra_igual(40_000, np.random.default_rng(99))
erro_bayes_i = np.mean((prob_bayes_igual(Xi_te) > 0.5).astype(int) != yi_te)
print(f"erro de Bayes com covariancias IGUAIS: {erro_bayes_i:.4f}")
print(f"(com covariancias diferentes era      : {erro_bayes:.4f})\n")

xx_i, yy_i = np.meshgrid(np.linspace(Xi[:, 0].min() - .8, Xi[:, 0].max() + .8, 320),
                         np.linspace(Xi[:, 1].min() - .8, Xi[:, 1].max() + .8, 320))
grade_i = np.c_[xx_i.ravel(), yy_i.ravel()]
Zb_i = (prob_bayes_igual(grade_i) > 0.5).reshape(xx_i.shape)

fig, axes = subplots(1, 3, figsize=(8.4, 3.0), sharex=True, sharey=True)
linhas_i = []
for ax, (nome, m) in zip(axes, modelos):
    m.fit(Xi, yi)
    Z = m.predict_proba(grade_i)[:, 1].reshape(xx_i.shape)
    ax.contourf(xx_i, yy_i, Z, levels=[0, 0.5, 1], colors=["steelblue", "crimson"],
                alpha=0.10)
    ax.contour(xx_i, yy_i, Zb_i, levels=[0.5], colors="green", linewidths=1.6,
               linestyles="--")
    ax.contour(xx_i, yy_i, Z, levels=[0.5], colors="black", linewidths=1.3)
    ax.scatter(Xi[yi == 0, 0], Xi[yi == 0, 1], s=6, color="steelblue", alpha=0.6)
    ax.scatter(Xi[yi == 1, 0], Xi[yi == 1, 1], s=6, color="crimson", alpha=0.6)
    ax.set_title(nome, fontsize=9)
    erro_i = 1 - m.score(Xi_te, yi_te)
    linhas_i.append({"classificador": nome, "erro no teste": erro_i,
                     "excesso sobre Bayes": erro_i - erro_bayes_i})

print(pd.DataFrame(linhas_i).set_index("classificador").round(4).to_string())

**O LDA passa a ser exato.** Excesso de $-0{,}0001$ sobre o piso de Bayes — zero, a
menos do ruído de medir em 40 mil pontos. E não é sorte: com $\Sigma_1=\Sigma_0$ os
termos quadráticos se cancelam na razão de densidades, a fronteira ótima é uma
**reta**, e o LDA é o modelo correto. É a mesma frase da Seção 3, com o QDA e o LDA
trocando de lugar.

**O QDA quase não piora: $+0{,}0015$.** Ele não está errado — o modelo dele
*contém* o do LDA como caso particular, bastando que as duas covariâncias estimadas
saiam parecidas. O que ele paga é variância: estima seis números de covariância em
vez de três, com os mesmos 400 pontos. Quinze centésimos de ponto percentual é o
preço da generalidade que aqui não era necessária.

**O Bayes ingênuo piora, e muito:** o excesso sobe de $+0{,}0355$ para $+0{,}0737$.
Igualar as covariâncias não conserta o problema dele, que é outro: a correlação de
$0{,}75$ agora está dentro das **duas** classes, e ele continua supondo zero. Com
covariância comum, a fronteira ótima é perpendicular a $\Sigma^{-1}(\mu_1-\mu_0)$,
e a correlação entra nessa direção por inteiro. Com as variâncias iguais a 1, como
aqui, zerar a correlação troca essa direção por $\mu_1-\mu_0$ — que fica a quase
**50 graus** da ótima.

---
## 5. `predict_proba`, e o corte que ninguém escolheu

Todo classificador do `scikit-learn` que estima probabilidades expõe dois métodos:
`predict_proba`, que devolve $\widehat P(Y=c\mid x)$, e `predict`, que devolve uma
classe. O segundo é o primeiro mais uma **decisão** — e a decisão padrão é cortar
em $0{,}5$.

In [ ]:
qda_aj = QDA().fit(X, y)
p = qda_aj.predict_proba(X_te)[:, 1]

print("predict == (predict_proba > 0.5)?",
      np.array_equal(qda_aj.predict(X_te), (p > 0.5).astype(int)))
print(f"\nprobabilidade estimada, 5 primeiras: {np.round(p[:5], 3)}")
print(f"probabilidade VERDADEIRA, as mesmas : {np.round(prob_bayes(X_te[:5]), 3)}")

fig, ax = subplots(figsize=(5.2, 3.0))
ax.scatter(prob_bayes(X_te[:4000]), p[:4000], s=3, alpha=0.25)
ax.plot([0, 1], [0, 1], color="black", lw=1)
ax.set_xlabel("P(Y=1|x) verdadeira"); ax.set_ylabel("estimada pelo QDA")
ax.set_title("probabilidade estimada x verdadeira (QDA, n=400)", fontsize=9)

O corte em $0{,}5$ minimiza o erro **quando os dois tipos de engano custam a
mesma coisa**. Quase nunca custam. Se um falso negativo é um tumor não detectado e
um falso positivo é um exame a mais, $0{,}5$ é uma escolha, e provavelmente uma
escolha ruim.

Guarde a separação que abriu o notebook: **estimar a probabilidade** e **decidir**
são duas etapas, e só a primeira é estatística. Este notebook cuidou dela; a Aula
08 cuida da segunda.

---
## 6. LDA ou QDA? A conta dos parâmetros

A regra de bolso diz: *"com poucos dados, prefira o LDA"*. O argumento é contagem
de parâmetros. Em $p$ dimensões, o LDA estima **uma** matriz de covariância —
$p(p+1)/2$ números — enquanto o QDA estima **uma por classe**, o dobro. Com duas
classes:

In [ ]:
for d in (2, 10, 50):
    p_lda = d * (d + 1) // 2 + 2 * d
    p_qda = 2 * (d * (d + 1) // 2) + 2 * d
    print(f"d = {d:3d}: LDA estima {p_lda:5d} parametros, QDA estima {p_qda:5d}"
          f"   (razao {p_qda/p_lda:.2f}x)")

Mais parâmetros, mais variância — o mesmo eixo da Aula 01. A regra de bolso, então,
prevê que o LDA ganha com $n$ pequeno e perde quando os dados chegam.

Vamos medir. E, o que é mais importante, medir em **duas dimensões diferentes**,
porque a regra só faz sentido se $p$ for grande o bastante para a conta acima
importar.

In [ ]:
_CACHE_Q = {}


def cov_fixa(d, qual):
    """Covariancia da classe `qual` em dimensao d -- sempre a mesma."""
    if (d, qual) not in _CACHE_Q:
        g = np.random.default_rng(1000 + qual)
        M = g.normal(size=(d, d))
        _CACHE_Q[(d, qual)] = (M @ M.T) / d + np.eye(d) * 0.5
    return _CACHE_Q[(d, qual)]


def gaussianas_dim(n, d, rng):
    n0 = n // 2
    mu = np.zeros(d)
    sinal = [1.6, 1.0, 0.7]                    # so as 3 primeiras coordenadas separam
    mu[:min(3, d)] = sinal[:min(3, d)]
    X = np.vstack([rng.multivariate_normal(np.zeros(d), cov_fixa(d, 0), size=n0),
                   rng.multivariate_normal(mu, cov_fixa(d, 1), size=n - n0)])
    return X, np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)


def comparar_lda_qda(sortear, ns, n_rep=60, semente=32):
    """Acuracia media de LDA e QDA em funcao de n, para um dado gerador."""
    rng_c = np.random.default_rng(semente)
    Xt, yt = sortear(20_000, np.random.default_rng(99))
    saida = {"LDA": np.zeros((n_rep, len(ns))), "QDA": np.zeros((n_rep, len(ns)))}
    for b in range(n_rep):
        for j, n in enumerate(ns):
            Xb, yb = sortear(int(n), rng_c)
            for nome, M in [("LDA", LDA), ("QDA", QDA)]:
                try:
                    saida[nome][b, j] = M().fit(Xb, yb).score(Xt, yt)
                except Exception:
                    saida[nome][b, j] = np.nan
    return {k: np.nanmean(v, axis=0) for k, v in saida.items()}


ns = np.array([20, 30, 50, 80, 150, 300, 700, 1500, 4000])
res10 = comparar_lda_qda(lambda n, r: gaussianas_dim(n, 10, r), ns)
res2 = comparar_lda_qda(amostra, ns)          # a populacao 2D da Secao 2

In [ ]:
tab = pd.DataFrame({"n": ns,
                    "d=10  LDA": res10["LDA"], "d=10  QDA": res10["QDA"],
                    "d=2   LDA": res2["LDA"], "d=2   QDA": res2["QDA"]}).set_index("n")

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.1), sharex=True)
for ax, res, d in [(ax1, res10, 10), (ax2, res2, 2)]:
    ax.plot(ns, res["LDA"], "o-", ms=4, color="steelblue", label="LDA")
    ax.plot(ns, res["QDA"], "s-", ms=4, color="crimson", label="QDA")
    ax.set_xscale("log"); ax.set_xticks(ns); ax.set_xticklabels(ns, fontsize=7)
    ax.set_xlabel("n (treino)"); ax.set_title(f"d = {d}", fontsize=9)
    ax.legend(fontsize=8)
ax1.set_ylabel("acuracia em 20 mil observacoes novas")

for d, res in [(10, res10), (2, res2)]:
    ganha = ns[res["QDA"] > res["LDA"]]
    print(f"d={d:2d}: o QDA ganha a partir de n = "
          f"{ganha.min() if len(ganha) else 'nunca'}")
tab.round(4)

> **A lição.** A regra de bolso está certa — mas só com duas condições que quase
> nunca são enunciadas junto.
>
> Em $p = 10$ ela se cumpre com folga. Com $n=30$ o LDA leva **cinco pontos
> percentuais** de vantagem, e o QDA só passa à frente a partir de $n=50$. Com
> $n=20$ o QDA nem aparece na tabela: são dez observações por classe em
> $\mathbb{R}^{10}$, a covariância amostral de cada classe tem posto no máximo 9, e
> o `scikit-learn` se recusa a ajustar o modelo. O `nan` é a conta de parâmetros
> cobrando antes de qualquer acurácia.
>
> Em $p = 2$ ela **não vale**: o QDA ganha já com 20 observações, e ganha em toda a
> faixa. Aqui não existe regime de "poucos dados" em que valha a pena impor uma
> covariância comum que é falsa.
>
> O motivo está na tabela de parâmetros. Em $p=2$ o QDA estima 10 números contra 7
> do LDA — três parâmetros a mais, que 20 observações pagam sem dificuldade. Em
> $p=10$ a diferença já é de 55 parâmetros, e aí o custo aparece.
>
> Formulação corrigida, com as duas condições: *o LDA ganha quando $n$ é pequeno
> **em relação aos parâmetros que o QDA estima a mais**, que crescem com $p^2$ — e o
> QDA só tem o que ganhar se as covariâncias das classes forem de fato diferentes.*
> Nesta população de $p=2$ elas são muito diferentes, com correlações de sinais
> opostos; numa em que fossem parecidas, o QDA teria pouco a cobrar, por mais barato
> que fosse.

---
## 7. Um caso real: diagnóstico de câncer de mama

O `breast_cancer` do `scikit-learn` traz 569 tumores descritos por 30 medidas de
imagem, classificados em benigno e maligno. É um problema pequeno, de dimensão
moderada e com sinal forte — bom para ver os três métodos lado a lado.

In [ ]:
from sklearn.datasets import load_breast_cancer

dados = load_breast_cancer()
Xr, yr = dados.data, dados.target
print(f"{Xr.shape[0]} tumores, {Xr.shape[1]} medidas")
print(f"classes: {dict(zip(dados.target_names, np.bincount(yr)))}")
print(f"prevalencia da classe 1 (benigno): {yr.mean():.3f}")

X_tr, X_ts, y_tr, y_ts = skm.train_test_split(Xr, yr, test_size=0.3,
                                              random_state=0, stratify=yr)

In [ ]:
candidatos = {
    "Bayes ingenuo": GaussianNB(),
    "LDA": LDA(),
    # reg_param poe um ridge minusculo na covariancia: sem ele o
    # scikit-learn recusa o ajuste, porque as 30 medidas sao quase colineares
    "QDA": QDA(reg_param=1e-4),
}

linhas = []
for nome, m in candidatos.items():
    m.fit(X_tr, y_tr)
    cv = skm.cross_val_score(m, X_tr, y_tr, cv=5, scoring="accuracy").mean()
    linhas.append({"classificador": nome, "acuracia (CV no treino)": cv,
                   "acuracia (teste)": m.score(X_ts, y_ts)})
linhas.append({"classificador": "chutar a classe majoritaria",
               "acuracia (CV no treino)": np.nan,
               "acuracia (teste)": max(y_ts.mean(), 1 - y_ts.mean())})
pd.DataFrame(linhas).set_index("classificador").round(4)

Note o que está e o que **não está** nessa tabela.

Está: os três chegam perto uns dos outros — de $0{,}9532$ do LDA a $0{,}9240$ do
Bayes ingênuo, no teste — e todos muito acima dos $0{,}6257$ de quem chuta a classe
majoritária.

Está também um aviso sobre ler diferenças pequenas. No teste, o LDA fica à frente do
QDA por $0{,}0117$; na validação cruzada, é o QDA que fica à frente, por $0{,}0025$.
As duas coisas convivem porque nenhuma das diferenças é grande: o conjunto de teste
tem 171 tumores, e $0{,}0117$ deles são **dois**. Com esse tamanho, LDA e QDA
empatam. A conta da Seção 6 existe — o QDA estima $30 \times 31/2 = 465$ números de
covariância **por classe**, a partir de menos de 400 observações de treino —, mas o
sinal deste banco é forte o bastante para ela não decidir a disputa.

Ela aparece, aliás, antes mesmo do resultado. Sem o `reg_param` da célula acima, o
`scikit-learn` se recusa a ajustar o QDA nestes dados — as 30 medidas de imagem são
colineares o bastante para a covariância de uma das classes ficar numericamente
singular. O aviso vem do próprio método, antes de qualquer acurácia.

O Bayes ingênuo, esse sim, fica atrás nas duas colunas: um ponto na validação
cruzada, três no teste. As 30 medidas são altamente correlacionadas entre si — raio,
perímetro e área do mesmo tumor —, e supor independência tem preço.

Não está: **nenhuma informação sobre que tipo de erro cada modelo comete.** Todos
os números são acurácia, e a acurácia trata "chamar de benigno um tumor maligno" e
"chamar de maligno um tumor benigno" como o mesmo evento. Em diagnóstico médico,
essa equivalência é indefensável — e neste conjunto a classe majoritária já dá 63%
de acurácia de graça, o que mostra o quanto essa métrica pode ser generosa.

É onde a Aula 08 começa.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| erro de Bayes | §2 | o piso da classificação, calculável só porque inventamos a população |
| plug-in | §3 | os três métodos diferem no modelo de $f_c(x)$ que plugam no Teorema de Bayes |
| QDA correto | §3 | quando a suposição é a verdade, o plug-in encosta no piso |
| covariâncias | §4 | o LDA impõe uma matriz só; o NB zera as correlações $+0{,}75$ e $-0{,}80$ |
| covariância comum | §4 | o LDA fica exato; o NB piora, porque zerar a correlação desvia a fronteira quase 50 graus |
| `predict_proba` | §5 | `predict` é `predict_proba > 0.5` — o corte é uma decisão, não um resultado |
| LDA × QDA | §6 | a regra "LDA com poucos dados" vale em $p=10$ (5 p.p. em $n=30$) e **falha** em $p=2$ |
| a conta certa | §6 | pesa $n$ contra os parâmetros extras, que crescem com $p^2$ — e se as covariâncias de fato diferem |
| caso real | §7 | LDA e QDA empatam dentro do ruído; o Bayes ingênuo paga por supor independência |
| o que falta | §7 | acurácia esconde *que tipo* de erro cada modelo comete |

**Leitura recomendada.** [AME] §7.1 (classificação, risco 0–1 e o classificador de
Bayes) e §8.1.3–8.1.4 (Bayes ingênuo e análise discriminante). [ISLP] §2.2.3 (o
classificador de Bayes e o erro de Bayes) e §4.4 (LDA, QDA e Bayes ingênuo — a
Figura 4.9 é a versão deles das nossas figuras das Seções 3 e 4, e o §4.4.3 traz a
conta de parâmetros da Seção 6).

**Para praticar.** `Lista teorica 07.pdf` (teórica, com gabarito) e
`Lista prática 07.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 08 pega a segunda metade da frase que abriu este notebook —
*"e depois decidir"* — e mostra que a acurácia é quase sempre a métrica errada,
especialmente quando uma das classes é rara.